# MSigDB ORA: study-subsampling coverage, 25% only (seed1)

**Environment:** `clamp-analyses`

Single-level slice of `00_bp_ora_analysis.ipynb`, scoped to **25% study
coverage** and **seed1 only** (preliminary results). Same method, same
cache paths (`rs{pct}_seed{{seed}}_msigdb.rds`) as the all-in-one notebook,
so `01_bp_coverage_plot.ipynb` reads this with no changes.

`pvalueCutoff = 0.05` (not `1`) — see the all-in-one notebook for why:
with no filtering, `enricher()` returns every one of 35k MSigDB terms
per LV (with a verbose `geneID` column), which OOM-killed a K=1728 run
at ~80GB RSS. `0.05` is the loosest FDR threshold used downstream and is
behavior-preserving for the coverage calculation.

In [ ]:
library(here)
library(clusterProfiler)
library(BiocParallel)

## Load MSigDB gene sets

In [ ]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
message(sprintf("MSigDB gene sets loaded: %d", length(unique(msig_gmt$term))))

## Paths

In [ ]:
models_dir <- here("output/01_model_building/04_archs4/07_bp_coverage_study")
output_dir <- here("output/03_model_biology/00_archs4/06_coverage_study/00_bp_ora_analysis")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(output_dir, "CLAMPbase"), recursive = TRUE, showWarnings = FALSE)

coverage  <- 25L
study_dir <- "03_bp_coverage_study_25"
seed      <- 1L  # seed1 only (preliminary)

seed_dir <- file.path(models_dir, study_dir, sprintf("study_coverage_rs%d_seed_%d", coverage, seed))

## Helper: run ORA for one model

Returns a list with raw `terms_padj` (minimum p.adjust per MSigDB term across all LVs).

In [ ]:
run_ora_for_model <- function(z_path, n_cores = 4) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  n_top          <- ceiling(0.01 * nrow(Z))
  n_lvs          <- ncol(Z)

  term_overlap   <- tapply(msig_gmt$gene %in% universe_genes, msig_gmt$term, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  top_genes_per_lv <- apply(Z, 2, function(lv) {
    universe_genes[order(lv, decreasing = TRUE)[seq_len(n_top)]]
  })

  # n_samples via B.csv header only (avoids loading the full multi-GB rds)
  b_path <- file.path(dirname(z_path), "B.csv")
  n_samples <- if (file.exists(b_path)) {
    ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
  } else NA_integer_

  bp <- BiocParallel::MulticoreParam(workers = n_cores, progressbar = FALSE)
  ora_results <- BiocParallel::bplapply(
    seq_len(n_lvs),
    function(i) {
      genes <- top_genes_per_lv[, i]
      tryCatch(
        clusterProfiler::enricher(
          gene          = genes,
          universe      = universe_genes,
          TERM2GENE     = msig_gmt,
          pAdjustMethod = "BH",
          pvalueCutoff  = 0.05,
          qvalueCutoff  = 1,
          minGSSize     = 10,
          maxGSSize     = 50000
        ),
        error = function(e) NULL
      )
    },
    BPPARAM = bp
  )

  all_dfs <- lapply(ora_results, function(r) {
    if (is.null(r) || nrow(as.data.frame(r)) == 0) return(NULL)
    as.data.frame(r)
  })
  all_dfs <- Filter(Negate(is.null), all_dfs)

  if (length(all_dfs) == 0) {
    warning("No ORA results returned for: ", z_path)
    return(NULL)
  }

  combined <- do.call(rbind, all_dfs)

  list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_top_genes    = n_top,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(combined$p.adjust, combined$ID, min)
  )
}

## Run ORA: CLAMPfull

In [ ]:
z_path     <- file.path(seed_dir, "CLAMPfull_hall", "Z.csv")
cache_path <- file.path(output_dir, "CLAMPfull", sprintf("rs%d_seed%d_msigdb.rds", coverage, seed))

sub_info_path <- file.path(seed_dir, "subsample_info.rds")
n_studies <- NA_integer_
if (file.exists(sub_info_path)) n_studies <- readRDS(sub_info_path)$n_studies

if (!file.exists(z_path)) {
  warning("Z.csv not found: ", z_path)
} else if (file.exists(cache_path)) {
  message(sprintf("Skipping CLAMPfull rs%d seed%d (cached)", coverage, seed))
} else {
  message(sprintf("Running ORA: CLAMPfull rs%d seed%d", coverage, seed))
  res <- run_ora_for_model(z_path)
  if (!is.null(res)) {
    res$n_studies <- n_studies
    saveRDS(res, cache_path)
    message(sprintf("  Saved: %s", basename(cache_path)))
  }
}

## Run ORA: CLAMPbase

In [ ]:
z_path     <- file.path(seed_dir, "CLAMPbase", "Z.csv")
cache_path <- file.path(output_dir, "CLAMPbase", sprintf("rs%d_seed%d_msigdb.rds", coverage, seed))

sub_info_path <- file.path(seed_dir, "subsample_info.rds")
n_studies <- NA_integer_
if (file.exists(sub_info_path)) n_studies <- readRDS(sub_info_path)$n_studies

if (!file.exists(z_path)) {
  warning("Z.csv not found: ", z_path)
} else if (file.exists(cache_path)) {
  message(sprintf("Skipping CLAMPbase rs%d seed%d (cached)", coverage, seed))
} else {
  message(sprintf("Running ORA: CLAMPbase rs%d seed%d", coverage, seed))
  res <- run_ora_for_model(z_path)
  if (!is.null(res)) {
    res$n_studies <- n_studies
    saveRDS(res, cache_path)
    message(sprintf("  Saved: %s", basename(cache_path)))
  }
}